<a href="https://colab.research.google.com/github/dbonacorsi/AML-Basic/blob/main/AMLBas_3_DataPrep_ExploreData_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explore your data with visualization

In this notebook, we will continue on **EDA** (**exploratory data analysis**): the process of analyzing and understanding your dataset before building models.

We will make:

* ***Univariate*** plots
   * Histograms
   * Density Plots
   * Box and Whisker Plots
* ***Multivariate*** plots

## Import the data

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/dbonacorsi/AML-Basic/refs/heads/main/datasets/pima-indians-diabetes.data.csv'

names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
data = pd.read_csv(url, names=names)
data

## Univariate plots

### Histograms

In [ ]:
from matplotlib import pyplot

In [ ]:
pyplot.rcParams["figure.figsize"] = [12,12]
data.hist()
pyplot.show()

---

### <font color=red>Exercise 1</font>

Compare what you see here, with what you saw as non-viz data exploration in the previous notebook. Where are you most familiar? The first, the second, or by using both?

Do you have additional observations from these plots, that you did not make over textual descriptive statistics? Hint: which histograms look roughly Gaussian and which are skewed?

---

### Density plots

In [ ]:
#from matplotlib import pyplot

In [ ]:
data.plot(kind='density', subplots=True, layout=(3,3), sharex=False)
pyplot.show()

### Box and Whisker plots

In [ ]:
data.plot(kind='box', subplots=True, layout=(3,3), sharex=False, sharey=False)
pyplot.show()

In [ ]:
### upload -> BoxAndWhiskers.png
from google.colab import files
uploaded = files.upload()

In [ ]:
from IPython.display import Image
Image(filename='/content/BoxAndWhiskers.png')

NOTE: Have you notices that the 0 placeholders from the previous notebook show up as outliers? **Visual investigation helps**!

## Multivariate plots
They show the interactions between multiple variables in your dataset.

### Correlation matrix plots

In [ ]:
#from matplotlib import pyplot
import numpy

In [ ]:
correlations = data.corr()
correlations

In [ ]:
# plot correlation matrix
fig = pyplot.figure()
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

# one tick per column, labels taken from the DataFrame itself
ticks = list(range(len(data.columns)))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(data.columns)
ax.set_yticklabels(data.columns)
pyplot.show()

Maybe:
*   choose more explanatory colors in the colormap;
*   display the correlation coefficients numerically

In [ ]:
# plot correlation matrix
fig = pyplot.figure(figsize=(9, 8))
ax = fig.add_subplot(111)
#cax = ax.matshow(correlations, vmin=-1, vmax=1)
cax = ax.matshow(correlations, vmin=-1, vmax=1, cmap='coolwarm')
cbar = fig.colorbar(cax)
cbar.set_label('Pearson correlation')

ticks = list(range(len(data.columns)))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(data.columns)
ax.set_yticklabels(data.columns)

# write each correlation value in its cell
for i in range(len(correlations)):
    for j in range(len(correlations.columns)):
        value = correlations.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha='center', va='center', fontsize=8,
                color='white' if abs(value) > 0.5 else 'black')

pyplot.show()

Look at the correlation matrix above. Notice that it also includes `class`, our target, not only the eight input features.

1. Which features have the strongest correlation with `class`? Are these the ones you would expect?
2. Which pairs of features are strongly correlated with each other, for example `age` and `preg`? Why might that matter when we build a model?
3. Should `class` be part of a plot used to choose input features? Why or why not?

Train yourself: target relationships first, then redundancy between inputs, then the modelling implication.

### Answers below

1. Strongest correlations with class: plas (glucose) is the strongest, at roughly 0.47, followed by mass (BMI) and age, both weaker. This is what you would expect: glucose and BMI are established diabetes risk factors.

2. Strongly correlated feature pairs: preg and age are the clearest (roughly 0.54), because older women have had more pregnancies. skin and mass are also moderately correlated. Redundancy matters because two near-duplicate inputs make coefficients unstable and split importance scores.

3. Should class guide feature choice? Only within a proper procedure. Choosing features by looking at the full correlation matrix uses the target before any evaluation, which biases the results. The target is for measuring model performance, and any selection based on it should happen inside validation, as discussed in one of next notebooks.


### Scatter plot matrix

In [ ]:
from pandas.plotting import scatter_matrix

In [ ]:
#scatter_matrix(data)
scatter_matrix(data, figsize=(12, 12), alpha=0.5)
pyplot.show()

### <font color=red>Exercise 2</font>

Do you spot something interesting in the data by visualizing it in this way?

This is the right time to discuss a bit about **feature correlations**.

The correlation matrix (both in numbers and in viz) shows how strongly pairs of features move together, and how each feature relates to the target `class`. But how is it counted? Pearson measures only linear relationships, so a weak value does not mean a feature is useless. Also, Pearson correlation is computed on the raw data, so e.g. the zeros in plas, pres, skin, test and mass distort the coefficients. Recompute the matrix after the zeros are handled, and treat any earlier conclusion as provisional.

In addition, high correlation *between features* (e.g. `skin` and `mass`) means they carry partly redundant information. It is not automatically a reason to drop one of the two, though. A correlation with `class` is what we want.

Consider that this aspet can impact models in different ways, dependingon the model. This can destabilise linear models and blur their coefficients, for example.
*   In linear models (e.g. logistic regression), correlated features make the individual coefficients unstable and hard to interpret. Small changes in the data can shift weight between the two features, and the standard errors inflate. Predictions are usually still fine, but “which feature matters” becomes unreliable.
*   Tree-based models: prediction is rarely hurt by correlated features. The problem is in importance scores: a tree can use either of two correlated features, so the importance gets split between them and each looks less important than it is.

So, before removing a feature, check whether the correlation is very high (say |r| > 0.9), whether the model actually needs it or benefits from it, and whether the correlation is distorted by any existing zero placeholders.

Feature selection is covered more in one of the next notebooks.



### <font color=red>Exercise 3</font>

Try to review previous and current notebooks, and think how to produce a new one of your own that would eventually digest all exploration-related tasks (both textually and visually), on a generic CSV file in input, in one shot. As a skeleton outline of a reusable EDA function:
1. input: a CSV path
2. steps: shape, dtypes, describe, class counts, correlations, skew, histograms, boxplots

Some hints:
* Stay clean
* Focus on instructions you need, avoid unnecessary ones
* be careful with heading levels (indentation)
* test it by trying `Runtime` -> `Run all` in the GUI menu

This is not wasted time: if you do it cleanly and usefully, you might end up in expand it further and re-use it often!

## Summary

What we did:

* **EDA** (exploratory data analysis) means analysing and understanding your dataset before building models.
* We used 5 quick ways to explore the dataset visually:
  1. **Histograms**: `data.hist()`, the distribution of each feature.
  2. **Density plots**: `data.plot(kind='density')`, a smoothed view of each distribution.
  3. **Box and whisker plots**: `data.plot(kind='box')`, the median, spread and outliers of each feature.
  4. **Correlation matrix**: `data.corr()` shown with `matshow`, the pairwise linear relationships.
  5. **Scatter plot matrix**: `scatter_matrix(data)`, pairwise scatter plots with histograms on the diagonal.
* Each of these would have taken considerable time to produce in a basic plotting system. Pandas and matplotlib make them one line each.

## What's next

Let's start to manipulate the data: we need to prepare the data to best expose the structure of your problem to modeling algorithms.